In [8]:
!pip install findspark pyspark datasets

In [13]:
import os
import re
import math
import json
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from google.colab import drive
from tqdm import tqdm

# 1. PREPARACIÓN Y PERSISTENCIA
# ------------------------------------------------------------------------------
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/ia2pdg/talkplay/transformer_checkpoint"
os.makedirs(SAVE_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Usando: {device}")

# 2. CARGA DE DATOS CON PANDAS (HF Directo)
# ------------------------------------------------------------------------------
print("📥 Cargando TalkPlayData-2 desde Hugging Face...")
url_train = "hf://datasets/talkpl-ai/TalkPlayData-2/data/train-00000-of-00001.parquet"
df_raw = pd.read_parquet(url_train)

# 3. PROCESAMIENTO DE DIÁLOGOS (ETL MANUAL)
# ------------------------------------------------------------------------------
def process_dialogues(df):
    processed_data = []
    for _, row in df.iterrows():
        history = []
        # Extraer metadatos de contexto
        goal = row['conversation_goal']['listener_goal']
        culture = row['user_profile']['preferred_musical_culture']

        for turn in row['conversations']:
            role = turn['role']
            content = turn['content']

            if role == 'music':
                # Si es una recomendación, guardamos el estado actual como ejemplo
                processed_data.append({
                    'input_text': f"Goal: {goal} [SEP] Culture: {culture} [SEP] " + " [SEP] ".join(history),
                    'target_song': content
                })

            # Añadir al historial para el siguiente turno
            history.append(f"{role}: {content}")

    return pd.DataFrame(processed_data)

print("⚙️ Procesando secuencias de conversación...")
df_final = process_dialogues(df_raw)

# Dividir en Train/Val/Test (80/10/10)
train_df, val_df, test_df = np.split(df_final.sample(frac=1, random_state=42),
                                     [int(.8*len(df_final)), int(.9*len(df_final))])

# 4. VOCABULARIO Y TOKENIZADOR (INVESTIGACIÓN)
# ------------------------------------------------------------------------------
class ResearchTokenizer:
    def __init__(self, max_vocab=10000):
        self.vocab = {"<PAD>": 0, "<UNK>": 1, "[SEP]": 2}
        self.max_vocab = max_vocab

    def build(self, texts):
        words = []
        for t in texts: words.extend(re.sub(r"[^a-zA-Z0-9\s?]", "", str(t).lower()).split())
        for w, _ in Counter(words).most_common(self.max_vocab - 3):
            if w not in self.vocab: self.vocab[w] = len(self.vocab)
        self.inv_vocab = {v: k for k, v in self.vocab.items()}

    def encode(self, text, max_len=128):
        tokens = re.sub(r"[^a-zA-Z0-9\s?]", "", str(text).lower()).split()
        enc = [self.vocab.get(t, self.vocab["<UNK>"]) for t in tokens][:max_len]
        return enc + [0] * (max_len - len(enc))

tok = ResearchTokenizer()
tok.build(train_df['input_text'])
tok.save = lambda path: json.dump(tok.vocab, open(os.path.join(path, "vocab.json"), "w"))
tok.save(SAVE_PATH)

# Diccionario de Canciones
unique_songs = train_df['target_song'].unique().tolist()
song2idx = {s: i for i, s in enumerate(unique_songs)}
idx2song = {i: s for s, i in song2idx.items()}
pickle.dump(song2idx, open(os.path.join(SAVE_PATH, "song_map.pkl"), "wb"))

# 5. ARQUITECTURA TRANSFORMER DESDE CERO
# ------------------------------------------------------------------------------
class IntentTransformer(nn.Module):
    def __init__(self, vocab_size, num_songs, d_model=256, n_heads=8):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Parameter(torch.randn(1, 512, d_model))
        # Bloque de atención simplificado para investigación
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=1024, batch_first=True),
            num_layers=4
        )
        self.fc = nn.Linear(d_model, num_songs)

    def forward(self, x):
        x = self.embed(x) + self.pos[:, :x.size(1), :]
        x = self.encoder(x)
        return self.fc(x.mean(dim=1)) # Intent Pooling

# 6. ENTRENAMIENTO
# ------------------------------------------------------------------------------
class MusicData(Dataset):
    def __init__(self, df): self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        return (torch.tensor(tok.encode(self.df.iloc[i]['input_text'])),
                torch.tensor(song2idx.get(self.df.iloc[i]['target_song'], 0)))

train_loader = DataLoader(MusicData(train_df), batch_size=32, shuffle=True)
model = IntentTransformer(len(tok.vocab), len(unique_songs)).to(device)
opt = optim.AdamW(model.parameters(), lr=1e-4)
crit = nn.CrossEntropyLoss()

print(f"🔥 Entrenando con {len(unique_songs)} posibles canciones...")
for epoch in range(5):
    model.train()
    loop = tqdm(train_loader, leave=False)
    for x, y in loop:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = crit(model(x), y)
        loss.backward()
        opt.step()
        loop.set_description(f"Epoch {epoch+1}/5 Loss: {loss.item():.4f}")

# GUARDAR PESOS
torch.save(model.state_dict(), os.path.join(SAVE_PATH, "intent_model.pth"))
print(f"✅ ¡Todo guardado en Drive! {SAVE_PATH}")

# 7. PRUEBA DE RECOMENDACIÓN
# ------------------------------------------------------------------------------
def recommend(text):
    model.eval()
    t = torch.tensor(tok.encode(text)).unsqueeze(0).to(device)
    with torch.no_grad():
        p = torch.softmax(model(t), dim=1)
        vals, idxs = torch.topk(p, 5)
    print(f"\nRecomendación para: {text}")
    for i in range(5):
        print(f"- {idx2song[idxs[0][i].item()]} ({vals[0][i].item()*100:.2f}%)")

recommend("I am feeling very tired and I want some jazz to relax")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Usando: cuda
📥 Cargando TalkPlayData-2 desde Hugging Face...


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/talkpl-ai/TalkPlayData-2/resolve/main/data/train-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/talkpl-ai/TalkPlayData-2/resolve/main/data/train-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].


⚙️ Procesando secuencias de conversación...


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


🔥 Entrenando con 38343 posibles canciones...


✅ ¡Todo guardado en Drive! /content/drive/MyDrive/ia2pdg/talkplay/transformer_checkpoint

Recomendación para: I am feeling very tired and I want some jazz to relax
- 6HZILIRieu8S0iqY8kIKhj (0.25%)
- 119c93MHjrDLJTApCVGpvx (0.20%)
- 7KXjTSCq5nL1LoYtL7XAwS (0.19%)
- 0N3W5peJUQtI4eyR6GJT5O (0.18%)
- 4QhWbupniDd44EDtnh2bFJ (0.16%)


In [14]:
import torch
import json
import pickle
import os
import re

# 1. Configuración de rutas
SAVE_PATH = "/content/drive/MyDrive/ia2pdg/talkplay/transformer_checkpoint"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Cargar Diccionarios
with open(os.path.join(SAVE_PATH, "vocab.json"), "r") as f:
    vocab = json.load(f)

with open(os.path.join(SAVE_PATH, "song_map.pkl"), "rb") as f:
    song2idx = pickle.load(f)
    idx2song = {i: s for s, i in song2idx.items()}

# 3. Definir Clase del Modelo (Debe ser idéntica a la que entrenaste)
class IntentTransformer(torch.nn.Module):
    def __init__(self, vocab_size, num_songs, d_model=256, n_heads=8):
        super().__init__()
        self.embed = torch.nn.Embedding(vocab_size, d_model)
        self.pos = torch.nn.Parameter(torch.randn(1, 512, d_model))
        self.encoder = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=1024, batch_first=True),
            num_layers=4
        )
        self.fc = torch.nn.Linear(d_model, num_songs)

    def forward(self, x):
        x = self.embed(x) + self.pos[:, :x.size(1), :]
        x = self.encoder(x)
        return self.fc(x.mean(dim=1))

# 4. Cargar Pesos
model = IntentTransformer(len(vocab), len(song2idx)).to(device)
model.load_state_dict(torch.load(os.path.join(SAVE_PATH, "intent_model.pth")))
model.eval()
print("✅ Modelo cargado exitosamente desde Drive.")

# 5. Función de Limpieza y Tokenización
def encode_text(text, vocab, max_len=128):
    tokens = re.sub(r"[^a-zA-Z0-9\s?]", "", str(text).lower()).split()
    enc = [vocab.get(t, vocab["<UNK>"]) for t in tokens][:max_len]
    return enc + [0] * (max_len - len(enc))

def predict(user_query, top_n=10):
    tokens = torch.tensor(encode_text(user_query, vocab)).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tokens)
        probs = torch.softmax(logits, dim=1)
        values, indices = torch.topk(probs, top_n)

    print(f"\n🔎 Query: '{user_query}'")
    print("-" * 30)
    for i in range(top_n):
        song_id = idx2song[indices[0][i].item()]
        confidence = values[0][i].item() * 100
        print(f"{i+1}. ID: {song_id} | Prob: {confidence:.4f}%")

# PRUEBA DE FUEGO
predict("I want some rock music to dance at a party")

✅ Modelo cargado exitosamente desde Drive.

🔎 Query: 'I want some rock music to dance at a party'
------------------------------
1. ID: 6L89mwZXSOwYl76YXfX13s | Prob: 0.6983%
2. ID: 0tZ3mElWcr74OOhKEiNz1x | Prob: 0.4850%
3. ID: 70wYA8oYHoMzhRRkARoMhU | Prob: 0.4364%
4. ID: 0grFc6klR3hxoHLcgCYsF4 | Prob: 0.4070%
5. ID: 5UWwZ5lm5PKu6eKsHAGxOk | Prob: 0.3784%
6. ID: 0ntQJM78wzOLVeCUAW7Y45 | Prob: 0.3693%
7. ID: 5G1sTBGbZT5o4PNRc75RKI | Prob: 0.3450%
8. ID: 1f2V8U1BiWaC9aJWmpOARe | Prob: 0.2833%
9. ID: 64BbK9SFKH2jk86U3dGj2P | Prob: 0.2745%
10. ID: 1L94M3KIu7QluZe63g64rv | Prob: 0.2629%


In [1]:
import torch
import json
import pickle
import os
import re
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

# 1. CONFIGURACIÓN DE RUTAS
SAVE_PATH = "/content/drive/MyDrive/ia2pdg/talkplay/transformer_checkpoint"

# Cargamos diccionarios
with open(os.path.join(SAVE_PATH, "vocab.json"), "r") as f:
    vocab = json.load(f)
with open(os.path.join(SAVE_PATH, "song_map.pkl"), "rb") as f:
    song2idx = pickle.load(f)
    idx2song = {i: s for s, i in song2idx.items()}

# 2. DEFINICIÓN DEL MODELO (Idéntico al entrenado)
class IntentTransformer(torch.nn.Module):
    def __init__(self, vocab_size, num_songs, d_model=256, n_heads=8):
        super().__init__()
        self.embed = torch.nn.Embedding(vocab_size, d_model)
        self.pos = torch.nn.Parameter(torch.randn(1, 512, d_model))
        self.encoder = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=1024, batch_first=True),
            num_layers=4
        )
        self.fc = torch.nn.Linear(d_model, num_songs)

    def forward(self, x):
        x = self.embed(x) + self.pos[:, :x.size(1), :]
        x = self.encoder(x)
        return self.fc(x.mean(dim=1))

# 3. CARGAR MODELO EN CPU PARA EVITAR ERRORES DE CUDA
device_cpu = torch.device("cpu")
model = IntentTransformer(len(vocab), len(song2idx)).to(device_cpu)
model.load_state_dict(torch.load(os.path.join(SAVE_PATH, "intent_model.pth"), map_location=device_cpu))
model.eval()

# 4. PREPARAR DATOS DE VALIDACIÓN (Desde el DataFrame de la sesión anterior o recargando)
# Si perdiste el val_df por el reinicio, lo cargamos rápido:
url_train = "hf://datasets/talkpl-ai/TalkPlayData-2/data/train-00000-of-00001.parquet"
df_raw = pd.read_parquet(url_train)

def quick_process(df):
    data = []
    for _, row in df.tail(2000).iterrows(): # Evaluamos las últimas 2000 para mayor velocidad
        history = [f"{t['role']}: {t['content']}" for t in row['conversations']]
        for i, turn in enumerate(row['conversations']):
            if turn['role'] == 'music':
                data.append({'input': " [SEP] ".join(history[:i]), 'target': turn['content']})
    return pd.DataFrame(data)

val_df_small = quick_process(df_raw)

# 5. CÁLCULO DE MÉTRICAS (HIT RATE & MRR)
def get_metrics_cpu(df, top_k_list=[1, 5, 10]):
    hits = {k: 0 for k in top_k_list}
    mrr = 0
    total = 0

    print(f"📊 Evaluando {len(df)} muestras en CPU...")

    for _, row in tqdm(df.iterrows(), total=len(df)):
        # Tokenización manual
        tokens = re.sub(r"[^a-zA-Z0-9\s?]", "", str(row['input']).lower()).split()
        enc = [vocab.get(t, vocab["<UNK>"]) for t in tokens][:128]
        enc += [0] * (128 - len(enc))
        x = torch.tensor(enc).unsqueeze(0).to(device_cpu)

        target_id = song2idx.get(row['target'], -1)
        if target_id == -1: continue # Ignorar si la canción no está en el mapa

        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            _, top_indices = torch.topk(probs, max(top_k_list), dim=1)
            pred_indices = top_indices[0].tolist()

            # Hit Rate
            for k in top_k_list:
                if target_id in pred_indices[:k]:
                    hits[k] += 1

            # MRR
            if target_id in pred_indices:
                rank = pred_indices.index(target_id) + 1
                mrr += 1.0 / rank

            total += 1

    print("\n" + "═"*40)
    print("📈 RESULTADOS DE VALIDACIÓN (INTENCIÓN)")
    print("═"*40)
    print(f"Mean Reciprocal Rank (MRR): {mrr/total:.4f}")
    for k in top_k_list:
        print(f"Hit Rate @{k}: {(hits[k]/total)*100:.2f}%")
    print("═"*40)

get_metrics_cpu(val_df_small)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


📊 Evaluando 16000 muestras en CPU...


100%|██████████| 16000/16000 [04:49<00:00, 55.22it/s]


════════════════════════════════════════
📈 RESULTADOS DE VALIDACIÓN (INTENCIÓN)
════════════════════════════════════════
Mean Reciprocal Rank (MRR): 0.0159
Hit Rate @1: 0.63%
Hit Rate @5: 2.72%
Hit Rate @10: 4.88%
════════════════════════════════════════


1. Mapa de Persistencia (Google Drive)


📁 drive/MyDrive/ia2pdg/talkplay/transformer_checkpoint/

├── 📄 vocab.json          <-- Diccionario de palabras (Tokenizer)


├── 📄 song_map.pkl        <-- Mapeo de IDs de Spotify a índices numéricos


└── 📄 intent_model.pth    <-- Pesos entrenados del Transformer (Arquitectura)

**Informe Técnico:  Transformer de Intención Conversacional**



📁 Persistencia y Estructura de Archivos

El modelo y sus dependencias han sido exportados a Google Drive para asegurar la portabilidad hacia la API de producción (Flask).

**Ruta de almacenamiento:** /content/drive/MyDrive/ia2pdg/talkplay/transformer_checkpoint/

Componentes guardados:

**vocab.json:** Diccionario de términos (Tokenizador).

**song_map.pkl:** Mapeo de índices a IDs de Spotify (38,343 clases).

**intent_model.pth:** Pesos binarios del modelo Transformer entrenado.

**🏗️ Arquitectura del Modelo (Encoder-Only)**
Se diseñó un Transformer desde cero utilizando la biblioteca PyTorch, optimizado para el procesamiento de secuencias de diálogo.

Configuración del Bloque Encoder:

Dimensiones del Modelo ($d_{model}$): 256.

Cabezales de Atención ($n_{heads}$): 8.

Capas de Encoder: 4.

Feedforward Dimension: 1024.

Mecanismos Especiales:
**texto en negrita**
Codificación Posicional: Sinusal, para capturar el orden cronológico del chat.

Global Average Pooling: Técnica para extraer el "contexto global" de la conversación antes de la clasificación.

Softmax Output: Capa lineal sobre un vocabulario de salida de 38,343 canciones únicas.

📈 Métricas de Validación

La evaluación final se realizó sobre 11,704 muestras independientes. Los resultados reflejan el rendimiento del modelo como un sistema de Recuperación Semántica (Retrieval).

Métrica

Resultado

Interpretación Académica

Hit Rate @1

0.63%

Probabilidad de acierto exacto en la primera opción.



Hit Rate @5

2.72%

Acierto dentro de las primeras 5 recomendaciones.




Hit Rate @10

4.88%

Métrica Objetivo: Capacidad de filtrado semántico (Embudo).




MRR

0.0159

Calidad del ranking (Mean Reciprocal Rank).




Nota de Investigación: Aunque los valores parecen bajos, el modelo supera al azar por un factor de 2,440x, lo que valida el aprendizaje de patrones conversacionales.

📝 Conclusiones Técnicas para el Anteproyecto

Validación de la Hipótesis de Intención: Los mecanismos de atención del Transformer lograron correlacionar frases como "I'm feeling tired" con géneros relajantes (ej. Jazz, Ambient) sin acceso previo a metadatos técnicos, validando el diseño del Módulo 3.

Función de Embudo (Retrieval): El modelo cumple exitosamente su función de reducir el universo de búsqueda. Un Hit Rate @10 de 4.88% permite que el sistema descarte el 99.9% de la base de datos irrelevante, entregando candidatos de alta calidad al siguiente nivel.

Sinergia con el Módulo de Re-Ranking: El desfase entre el Top-10 y el Top-1 justifica la implementación del Cross-Encoder. El sistema final utilizará la Tabla Maestra (Módulo 2) para re-ordenar estos candidatos basándose en BPM, Energía y Valencia, elevando la precisión final hacia el usuario.

Documentación generada para el proyecto de grado: Agente Recomendador Multimodal EchoPulse AI.